# Feature Engineering : News embedding

The objective of this stage is to convert the textual data (News) from a specific day into a unique numerical representation that captures the global market sentiment for that day.
- **Lexicon Filtering**: For each news article, only the words included in the time-aware, domain-specific lexicon generated in the previous step are retained.
- **Word Embedding Integration**: Each remaining word is replaced by its corresponding Word2Vec/Bert dense vector representation.
- **Document Representation**: The final vector for each article is constructed by computing the average of the word-embeddings of its filtered words, resulting in a unique news-embedding.

In [7]:
import pandas as pd
import numpy as np
import os
import sys
import gensim.downloader as api
from sentence_transformers import SentenceTransformer

sys.path.append(os.path.abspath(os.path.join("..")))
from src.feature_engineering import *

## Word2Vec Model

### Loading Word2Vec pre-trained embeddings

In [10]:
# Word2Vec (Google News 300D)
word_vectors = api.load("word2vec-google-news-300")
print(f"Vectors dimensions : {word_vectors.vector_size}")
print(f"# Known words : {len(word_vectors)}")

Vectors dimensions : 300
# Known words : 3000000


### News embeddings procedure

In [11]:
# Création d'embeddings pour chaque article de news
news = pd.read_csv("../data/processed/news_2026_spacy.csv")
news_features = run_feature_engineering_pipeline(
    news, "../data/processed/daily_lexicons_filtered_spacy/", word_vectors
)
news_features.to_csv("../data/for_models/news_features_w2v.csv", index=False)

Feature Engineering: 100%|██████████| 90/90 [00:00<00:00, 180.07it/s]


In [12]:
X = np.stack(news_features["embedding"].values)

## DistilRoBERTa

### Loading DistilRoBERTa pre-trained model

In [13]:
print("Loading DistilRoBERTa model...")
model_bpe = SentenceTransformer("sentence-transformers/all-distilroberta-v1")
print(model_bpe)

Loading DistilRoBERTa model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2750.61it/s]
RobertaModel LOAD REPORT from: sentence-transformers/all-distilroberta-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'RobertaModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


### News embeddings procedure

In [14]:
# --- Execution ---
# Load your BPE preprocessed news
news = pd.read_csv("../data/processed/news_2026_bpe.csv")

In [15]:
# Generate embeddings (batch_size=16 est parfait pour éviter de surcharger le CPU)
news_features = run_document_embedding_bpe(news, model_bpe, batch_size=32)

# Save result
news_features.to_csv("../data/for_models/news_features_bpe.csv", index=False)
print(f"Success! {len(news_features)} documents embedded.")

Starting batch embedding for 736 articles...


Batches: 100%|██████████| 23/23 [07:46<00:00, 20.29s/it]


Success! 736 documents embedded.


### Comparaison des modèles d'Embeddings

Voici un tableau comparatif résumant les différences fondamentales entre ces deux approches, en s'appuyant sur leurs architectures respectives :

| Caractéristique | Word2Vec (`google-news-300`) | DistilRoBERTa (`all-distilroberta-v1`) |
| :--- | :--- | :--- |
| **Type d'architecture** | Réseau de neurones simple (Statique) | Transformer (Contextuel) |
| **Dimension des vecteurs** | 300 | 768 |
| **Tokenisation** | Mots entiers (Séparation par espaces) | Sous-mots / BPE (Byte-Pair Encoding) |
| **Taille du vocabulaire** | 3 000 000 de mots figés | ~50 000 sous-mots (capable d'en créer une infinité) |
| **Limite de lecture** | Aucune limite (traite mot par mot) | **512 tokens** maximum d'un coup |
| **Gestion du contexte** | **Non.** Le mot "vol" a le même vecteur, qu'il s'agisse d'un oiseau ou d'un vol de banque. | **Oui.** Le vecteur de "vol" s'adapte selon les autres mots de la phrase. |
| **Opération finale** | Moyenne manuelle des vecteurs des mots de l'article. | Pooling automatique (`mean_tokens`) + Normalisation intégrée. |

#### En résumé pour le projet :
* **Word2Vec** est un énorme dictionnaire de traduction géométrique. Il est extrêmement rapide mais limité si un mot est hors de son vocabulaire ou possède plusieurs sens.
* **DistilRoBERTa** lit les 512 premiers jetons de l'article, comprend les relations mathématiques *entre* les mots de cette phrase précise, et génère un vecteur très riche (768D) qui résume parfaitement le sens global du texte.